In [4]:
# Instalación de librerías
!pip install -q -U langchain langchain-text-splitters langchain-huggingface langchain-chroma langchain-community pypdf sentence-transformers chromadb faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 3.4 MB/s eta 0:00:00


In [9]:
!mkdir -p /content/documents

# Parte 1. Preparación del entorno

In [17]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline

print("Entorno listo ✅")
from google.colab import userdata
os.environ["RAG"] = userdata.get('RAG')

Entorno listo ✅


# Parte 2. Construcción de la base documental

In [12]:
document_path = "/content/documents/"
documentos = []
for archivo in pdfs:
    ruta = os.path.join(document_path, archivo)
    loader = PyPDFLoader(ruta)
    paginas = loader.load()
    documentos.extend(paginas)
    print(f"✅ Cargado: {archivo} ({len(paginas)} páginas)")

print(f"\nNúmero total de documentos (páginas) cargados: {len(documentos)}")

✅ Cargado: poa_2025_mas_resolución0046180001777305442.pdf (87 páginas)
✅ Cargado: poa_2024_mas_resolución0207210001777305442.pdf (52 páginas)
✅ Cargado: poa-2026_mas_resolucion-aprobación0405566001777305441.pdf (253 páginas)

Número total de documentos (páginas) cargados: 392


In [13]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

fragmentos = splitter.split_documents(documentos)
print(f"Número de fragmentos generados: {len(fragmentos)}")
print("Ejemplo de fragmento:\n", fragmentos[0].page_content[:300])

Número de fragmentos generados: 1707
Ejemplo de fragmento:
 1 
 
RESOLUCIÓN N° 008-R-UNL-2025 
Dr. Nikolay Aguirre 
RECTOR DE LA UNIVERSIDAD NACIONAL DE LOJA  
 
CONSIDERANDO: 
 
PRIMERO.- El Art. 226 de la Constitución de la República del Ecuador, establece que 
"Las instituciones del Estado, sus organismos, dependencias, las servidoras o 
servidores públic


 # Parte  3. Generación de embeddings

In [18]:
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Prueba rápida
vector_prueba = modelo_embeddings.embed_query("Plan Operativo Anual")
print(f"Dimensión del vector de embedding: {len(vector_prueba)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dimensión del vector de embedding: 384


 # Parte 4. Construcción de la base vectorial

In [19]:
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings,
    collection_name="poa_unl"
)

print(f"Base vectorial creada con {vectorstore._collection.count()} fragmentos indexados")

Base vectorial creada con 1707 fragmentos indexados


In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

nombre_modelo = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
modelo_llm = AutoModelForCausalLM.from_pretrained(nombre_modelo, torch_dtype=torch.float32)

def generar_respuesta(prompt_texto, max_new_tokens=300):
    mensajes = [
        {"role": "system", "content": "Eres un asistente que responde en español de forma clara y concisa, basándote únicamente en el contexto proporcionado."},
        {"role": "user", "content": prompt_texto}
    ]
    texto_formateado = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(texto_formateado, return_tensors="pt", truncation=True, max_length=2048)
    outputs = modelo_llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0
    )
    respuesta_completa = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extraer solo la parte generada (después del prompt)
    respuesta = respuesta_completa.split(texto_formateado.split("<|im_start|>assistant")[0] if "<|im_start|>" in texto_formateado else "")[-1]
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Prueba rápida
print(generar_respuesta("¿Qué es un Plan Operativo Anual?"))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Un Plan Operativo Anual es una estrategia para mejorar la eficiencia operativa de una organización. Es un documento que se utiliza para evaluar y planificar acciones necesarias para aumentar la productividad y reducir costos operativos.


#   Parte 5. Implementación del sistema RAG

In [32]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

print("Retriever creado correctamente.")

Retriever creado correctamente.


In [33]:
prompt_template = """
Eres un asistente especializado en responder preguntas únicamente con la información del contexto.

Reglas:
- Usa exclusivamente la información del contexto.
- No inventes información.
- No utilices conocimientos externos.
- Si el contexto no contiene la respuesta exacta, responde:
  "No encontré esa información en los documentos."
- Responde en español de forma clara.

================ CONTEXTO ================

{contexto}

==========================================

Pregunta:
{pregunta}

Respuesta:
"""

In [34]:
def preguntar_rag(pregunta, mostrar_fragmentos=True):

    # Recuperar documentos similares
    documentos = retriever.invoke(pregunta)

    # Unir el contenido de los documentos
    contexto = "\n\n".join(
        [doc.page_content for doc in documentos]
    )

    # Construir el prompt
    prompt = prompt_template.format(
        contexto=contexto,
        pregunta=pregunta
    )

    # Generar respuesta con Qwen
    respuesta = generar_respuesta(prompt)

    # Mostrar los fragmentos recuperados
    if mostrar_fragmentos:

        print("="*80)
        print("FRAGMENTOS RECUPERADOS")
        print("="*80)

        for i, doc in enumerate(documentos):

            print(f"\nFragmento {i+1}")
            print("-"*60)
            print(doc.page_content[:700])

    return respuesta

 # Parte 6. Pruebas del sistema

In [35]:
pregunta = "¿Qué es un Plan Operativo Anual?"

respuesta = preguntar_rag(pregunta)

print("\n")
print("="*80)
print("RESPUESTA DEL MODELO")
print("="*80)
print(respuesta)

FRAGMENTOS RECUPERADOS

Fragmento 1
------------------------------------------------------------
El Plan Operativo Anual se elabora, implementa y monitorea, al amparo de la normativa 
nacional e institucional, que se resume a continuación: 
 
El Art. 97, del Código Orgánico de Planificación y Finanzas Públicas (COPFP) establece que 
la Programación Presupuestaria es una fase del ciclo presupuestario y que: 
 
En base de los objetivos determinados por la planificación y las disponibilidades 
presupuestarias coherentes con el escenario fiscal esperado, se definen los 
programas, proyectos y actividades a incorporar en el presupuesto, con la 
identificación de las met as, los recursos necesarios, los impactos o resultados 
esperados de su entrega a la sociedad; y los plazos para su ejecuci

Fragmento 2
------------------------------------------------------------
control, seguimiento y evaluación del plan plurianual institucional y de los planes 
operativos anuales, los cuales considerarán

In [36]:
pregunta = "¿Quién aprueba el Plan Operativo Anual?"

respuesta = preguntar_rag(pregunta)

print("\nRespuesta:")
print(respuesta)

FRAGMENTOS RECUPERADOS

Fragmento 1
------------------------------------------------------------
4 
 
RESUELVO: 
 
Art. 1. - Aprobar el Plan Operativo Anual 2025 Institucional, de Facultades y 
Unidades Académicas y Administrativas de la Universidad Nacional de Loja (POA 
2025), cuyo detalle se encuentra en los anexos que forman parte integral de la 
presente Resolución. 
 
Art. 2.- Encargar a la Dirección de Planificación y Desarrollo, notificar la presente 
resolución de aprobación del Plan Operativo Anual 2025 Institucional, de 
Facultades y Unidad es Académicas y Administrativas de la Universidad Nacional 
de Loja (POA 2025) a las entidades públicas del Estado que corresponda, así como 
a las dependencias y unidades académicas y administrativas de la institución. 
 
Art. 3.- Dispone

Fragmento 2
------------------------------------------------------------
seguimiento a
graduados por
carrera y Plan de
Mejoras
CASTILLO CALDERON
JAIRO DARIO, CORONEL
ROMERO EDISON
LEONARDO, TAMBO
ENCAL

In [37]:
pregunta = "¿Cuáles son los componentes que debe contener un Plan Operativo Anual?"

respuesta = preguntar_rag(pregunta)

print("\nRespuesta:")
print(respuesta)

FRAGMENTOS RECUPERADOS

Fragmento 1
------------------------------------------------------------
El Plan Operativo Anual se elabora, implementa y monitorea, al amparo de la normativa 
nacional e institucional, que se resume a continuación: 
 
El Art. 97, del Código Orgánico de Planificación y Finanzas Públicas (COPFP) establece que 
la Programación Presupuestaria es una fase del ciclo presupuestario y que: 
 
En base de los objetivos determinados por la planificación y las disponibilidades 
presupuestarias coherentes con el escenario fiscal esperado, se definen los 
programas, proyectos y actividades a incorporar en el presupuesto, con la 
identificación de las met as, los recursos necesarios, los impactos o resultados 
esperados de su entrega a la sociedad; y los plazos para su ejecuci

Fragmento 2
------------------------------------------------------------
control, seguimiento y evaluación del plan plurianual institucional y de los planes 
operativos anuales, los cuales considerarán

In [38]:
pregunta = "¿Cuál es la finalidad del seguimiento y evaluación del Plan Operativo Anual?"

respuesta = preguntar_rag(pregunta)

print("\nRespuesta:")
print(respuesta)

FRAGMENTOS RECUPERADOS

Fragmento 1
------------------------------------------------------------
informe de
seguimiento a la
ejecución del Plan de
sostenibilidad
GALVEZ MAZA
JORGE RAMIRO 
X Informe de
seguimiento a la
ejecución del Plan 
E17.2. Innovación
del ciclo de la
planificación
institucional con
énfasis en la
articulación. 
OO17.11. Fortalecer la
planificación,
seguimiento y
evaluación
institucional. 
Número de eventos de
capacitación para la
planificación operativa,
seguimiento y
evaluación institucional
en el módulo de
planificación del SIAAF,
ejecutados. 
S1 y S2: Reporte
consolidado de
capacitaciones
realizadas por
semestre.
DPD 2 1 1 Organización y
ejecución de los
eventos de
capacitación
JADAN ORTEGA
JACQUELINE DEL
ROCIO 
X X Agenda de
capacitación sobre
procesos de
planifi

Fragmento 2
------------------------------------------------------------
3 
 
Desarrollo vigente y las prioridades contenidas en el Plan Estratégico de 
Desarrollo Institucional (PEDI 2024-2028) UNL So

In [40]:
pregunta = "¿Con qué documento institucional debe estar alineado el Plan Operativo Anual 2026 de la UNL?"

respuesta = preguntar_rag(pregunta)

print("\nRespuesta:")
print(respuesta)

FRAGMENTOS RECUPERADOS

Fragmento 1
------------------------------------------------------------
5. Despliegue del  
POA Institucional 2026

Fragmento 2
------------------------------------------------------------
1 
 
 
 
 
 
 
Universidad Nacional de Loja 
 
Plan Operativo Anual 2024 
Institucional y de Facultades 
 
 
 
 
 
 
 
 
Mayo 2024

Fragmento 3
------------------------------------------------------------
1 
Universidad Nacional de Loja  
    Plan Operativo Anual 2026 Institucional, de Facultades y Unidades Académicas y Administrativas   POA 2026      Febrero 2026

Fragmento 4
------------------------------------------------------------
6. Despliegue del  
POA de Facultades 2026

Fragmento 5
------------------------------------------------------------
de fecha 19 de marzo de 2024. 
 
Nuestra universidad , comprometida con el cumplimiento de los objetivos 
estratégicos definidos en el PEDI UNL Sostenible 2024 – 2028, ha desarrollado el proceso 
de construcción participativa de

# Parte 7. Análisis de resultados

Analizar  y  comparar  cómo  influye  la  calidad  de  los  documentos  sobre  el  desempeño del sistema sobre:

-  Calidad de las respuestas


Las respuestas generadas fueron, en su mayoría, coherentes con la información recuperada por el sistema RAG, permitiendo responder adecuadamente las consultas relacionadas con el Plan Operativo Anual. No obstante, se observaron algunos casos en los que el modelo interpretó el contexto de forma parcial o incorporó información no presente en los documentos. Por ejemplo, en la quinta consulta, aunque los fragmentos recuperados indicaban que el POA debía estar alineado con el Plan Estratégico de Desarrollo Institucional (PEDI 2024–2028), la respuesta se limitó a mencionar "POA Institucional 2026", omitiendo el elemento principal de la pregunta. Asimismo, en otras consultas el modelo generó explicaciones generales o añadió datos que no se encontraban explícitamente en los documentos recuperados. Estos resultados muestran que la calidad de la respuesta depende tanto de la relevancia de los fragmentos recuperados como de la capacidad del modelo para utilizar únicamente el contexto proporcionado.


-  Pertinencia de los documentos recuperados


En la mayoría de las consultas, los fragmentos recuperados estuvieron relacionados con la pregunta planteada y proporcionaron el contexto necesario para responder. No obstante, en algunas búsquedas también se recuperaron fragmentos de diferentes versiones del POA (2024, 2025 y 2026), lo que introdujo información adicional que no siempre era relevante para la consulta específica.


-  Posibles errores de recuperación

Se identificó que el sistema, en ocasiones, recuperó fragmentos de documentos con información similar pero correspondiente a distintos años, lo que ocasionó respuestas con datos inconsistentes, como referencias a un año diferente o la inclusión de nombres no solicitados. Esto evidencia que la calidad y actualidad de los documentos indexados influyen directamente en la precisión del sistema RAG.